In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.integrate
from matplotlib import rc

import optimal_consumption as oc

In [ ]:
rc("font", family="serif", serif=["Computer Modern"], size=18)
rc("text", usetex=True)
rc("axes.formatter", useoffset=False)

In [ ]:
R = 1.5


def delta(y):
    return 0.02


def r(y):
    return y


def lmbda(y):
    return 0.069 / 0.18


def sigma(y):
    return 0.18


rho = -0.0012

kappa = 0.43


def a(y):
    return -kappa * (y - 0.013)


def b(y):
    return np.full_like(y, 0.033, dtype=float)

In [ ]:
delta(0) / (1 - R) - lmbda(0) ** 2 / (2 * R)

In [ ]:
eta = oc.myopic_consumption_rate(R, r, delta, lmbda)

In [ ]:
X_sol, sol = oc.optimal_consumption_rate(R, r, delta, lmbda, rho, a, b, -50, 75, 10000)

In [ ]:
u = sol
u_prime = np.gradient(u, X_sol, edge_order=2)
u_prime2 = np.gradient(u_prime, X_sol, edge_order=2)

In [ ]:
X = np.linspace(-30, 50, 1000)
plt.plot(X, np.interp(X, X_sol, u_prime / u), label="$\\frac{u'}{u}$")
plt.axhline(
    (R - 1) / (R * kappa),
    color="grey",
    linestyle="dashed",
    label="$q = \\frac{R-1}{R\\kappa}$",
)
plt.legend()
plt.xlabel("y")
plt.savefig("img/vasicek/gradient.pdf", bbox_inches="tight")

In [ ]:
X = np.linspace(-5, 5, 100)
sol_pf = oc.optimal_investment_fraction(R, sigma, lmbda, rho, b, X_sol, sol)

plt.plot(X, 100 * np.interp(X, X_sol, sol_pf), label="Optimal")
plt.plot(
    X,
    100 * np.full_like(X, oc.myopic_investment_fraction(R, sigma, lmbda)(X)),
    label="Myopic",
)
plt.fill_between(
    X,
    100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X),
    100 * np.interp(X, X_sol, sol_pf),
    color="green",
    alpha=0.3,
    hatch="//",
    label="Intertemporal Hedge",
)

plt.legend()

plt.xlabel("$y$")
plt.ylabel("Risky Asset Allocation (\\%)")

plt.ylim(141.9, 142.5)

plt.savefig("img/vasicek/portfolio_limit.pdf", bbox_inches="tight")

In [ ]:
X = np.linspace(-5, 50, 1000)

plt.plot(X, 100 * (np.interp(X, X_sol, sol) - eta(X)), label="$\\xi_h$")
plt.axhline(100 * (-kappa), color="grey", linestyle="dashed", label="$-\\kappa$")

plt.axvline(
    delta(0) / (1 - R) - lmbda(0) ** 2 / (2 * R),
    color="grey",
    linestyle="dotted",
    label="$y^*$",
)

plt.legend()

plt.xlabel("$y$")
plt.ylabel("Consumption Adjustment (\%)")

plt.savefig("img/vasicek/consumption_limit.pdf", bbox_inches="tight")

In [ ]:
X = np.linspace(-0.06, 0.08, 1000)

plt.plot(X, 100 * np.interp(X, X_sol, sol), label="Optimal")
plt.plot(X, 100 * eta(X), label="Myopic")
plt.legend()
plt.xlabel("$y$")
plt.ylabel("Consumption Rate (\\%)")
plt.savefig("img/vasicek/consumption.pdf", bbox_inches="tight")
plt.show()

plt.plot(X, 100 * np.interp(X, X_sol, sol_pf), label="Optimal")
plt.plot(
    X,
    np.full_like(X, 100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X)),
    label="Myopic",
)
plt.fill_between(
    X,
    100 * oc.myopic_investment_fraction(R, sigma, lmbda)(X),
    100 * np.interp(X, X_sol, sol_pf),
    color="green",
    alpha=0.3,
    hatch="//",
    label="Intertemporal Hedge",
)
plt.ylabel("Risky Asset Allocation (\\%)")
plt.legend()
plt.xlabel("$y$")
plt.gca().get_xaxis().get_major_formatter().set_useOffset(False)
plt.ylim(141.9, 142.5)
plt.savefig("img/vasicek/portfolio.pdf", bbox_inches="tight")
plt.show()

## Consistency check with Guasoni & Wang

In [ ]:
b_gw = 0.43
theta = 0.013
gamma = 1.5
c = (gamma - 1) / gamma
a_gw = 0.033
rho = -0.0012
mu = 0.069
sigma_gw = 0.18
beta = 0.02

In [ ]:
k = b_gw * theta + ((1 - gamma) * sigma_gw * rho * a_gw * mu / sigma_gw**2) / gamma
A = a_gw**2

In [ ]:
def B(t):
    return (c * (1 - np.exp(-b_gw * t))) / b_gw


def C_p(t):
    return (
        -1
        / (4 * b_gw**3)
        * (
            c**2 * A * np.exp(-2 * b_gw * t)
            - (4 * c**2 * A - 4 * c * b_gw * k) * np.exp(-b_gw * t)
            + 3 * c**2 * A
            - 4 * c * b_gw * k
        )
        + (
            (c**2 * A) / (2 * b_gw**2)
            - ((gamma - 1) * mu**2 / sigma_gw**2) / (2 * gamma**2)
            - beta / gamma
            + ((gamma - 1) * (1 - rho**2) * A * c**2) / (2 * b_gw**2)
            - (c * k) / b_gw
        )
        * t
    )


def lower_bound(y):
    return 1 / scipy.integrate.quad(lambda t: np.exp(C_p(t) - B(t) * y), 0, 1000)[0]


def C_d(t):
    return (
        -1
        / (4 * b_gw**3)
        * (
            c**2 * A * np.exp(-2 * b_gw * t)
            - (4 * c**2 * A - 4 * c * b_gw * k) * np.exp(-b_gw * t)
            + 3 * c**2 * A
            - 4 * c * b_gw * k
        )
        + (
            (c**2 * A) / (2 * b_gw**2)
            - ((gamma - 1) * mu**2 / sigma_gw**2) / (2 * gamma**2)
            - beta / gamma
            - (c * k) / b_gw
        )
        * t
    )


def upper_bound(y):
    return 1 / scipy.integrate.quad(lambda t: np.exp(C_d(t) - B(t) * y), 0, 1000)[0]

In [ ]:
plt.plot(
    X := np.linspace(-0.06, 0.08, 100),
    np.vectorize(upper_bound)(X) * 100,
    color="gray",
    linestyle="dashed",
    label="A priori bounds",
)
plt.plot(X, np.interp(X, X_sol, sol) * 100, label="ODE Solution")
plt.plot(X, np.vectorize(lower_bound)(X) * 100, color="grey", linestyle="dashed")
plt.legend()
plt.xlabel("$y$")
plt.ylabel("Consumption Rate (\\%)")
plt.savefig("img/vasicek/apriori.pdf", bbox_inches="tight")